# Train CUB-200-2011 on the official splitProduces the first accuracy number from this project that can actually be defended.**Why retrain at all.** The checkpoint currently in the repo has no recorded training split, and measurement says it saw everything: 94.6% on official CUB *train*, 94.2% on official *test*, and **95.2% on official-test images that were never held out** — images it supposedly never saw scoring *higher* than ones it did, against a realistic ~86–88% ceiling for this architecture. Its numbers are memorisation, not skill.This notebook trains from `data/splits/official_*.txt`, whose sha256 hashes are written into the checkpoint, so every future claim can name the exact image list behind it.**Expected: 84–88% top-1 on official CUB test.** Anything above 90% from this recipe is a leak to investigate, not a win.Runtime: **Runtime → Change runtime type → T4 GPU**. About 2–4 hours for 35 epochs.

In [ ]:
import torch, subprocessprint(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],                     capture_output=True, text=True).stdout.strip() or "NO GPU")assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"print("torch", torch.__version__, "| cuda", torch.version.cuda)

## 1. Get the datasetCUB-200-2011 is 1.1 GB. Downloading it fresh each session takes ~5 minutes; caching theextracted copy on Drive makes later sessions instant. Pick whichever cell suits you.

In [ ]:
# Option A - download from Caltech (no Drive needed, ~5 min)import osif not os.path.exists("/content/CUB_200_2011"):    !wget -q --show-progress -O /content/CUB.tgz \        https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz    !tar -xzf /content/CUB.tgz -C /content/    !rm /content/CUB.tgzprint(sorted(os.listdir("/content/CUB_200_2011"))[:8])

In [ ]:
# Option B - reuse a copy already on your Drive (skip if you ran Option A)# from google.colab import drive; drive.mount('/content/drive')# !tar -xzf /content/drive/MyDrive/CUB_200_2011.tgz -C /content/

## 2. Bring in the project filesUpload `train.py`, `scripts/make_split.py` and — importantly — `data/splits/`.Reuse the **committed** split manifests rather than regenerating them. Regenerating with adifferent seed would silently produce a different train/val partition, and the whole point ofcommitting them is that the checkpoint's recorded hashes match what is in your repo.

In [ ]:
import osos.makedirs("/content/project/scripts", exist_ok=True)os.makedirs("/content/project/data/splits", exist_ok=True)os.chdir("/content/project")!ln -sfn /content/CUB_200_2011 /content/project/CUB_200_2011from google.colab import filesprint("Upload: train.py, scripts/make_split.py, and data/splits/official_*.txt + split_meta.json")up = files.upload()for name in up:    if name.startswith("official_") or name == "split_meta.json":        os.replace(name, f"data/splits/{name}")    elif name == "make_split.py":        os.replace(name, "scripts/make_split.py")print(sorted(os.listdir("data/splits")))

In [ ]:
# Verify the manifests are the ones you committed, before spending GPU hours on them.import hashlib, json, osmeta = json.load(open("data/splits/split_meta.json"))ok = Truefor split, expected in meta["sha256"].items():    p = f"data/splits/official_{split}.txt"    got = hashlib.sha256(open(p,"rb").read()).hexdigest()    n = sum(1 for _ in open(p))    print(f"{split:<6} {n:>5} images  {'MATCH' if got==expected else 'MISMATCH'}")    ok &= got == expectedassert ok, "Manifest hashes differ from split_meta.json - re-upload, do not regenerate"print("\nseed:", meta["seed"], "| val_frac:", meta["val_frac"])

## 3. Train`--workers 2` matches Colab's CPU allocation; higher values usually slow it down.Training runs at 300 px and evaluates at 380 px (FixRes) — faster per epoch *and* typicallyslightly more accurate than training and testing at the same size.

In [ ]:
!python train.py --epochs 35 --batch-size 32 --workers 2 \                 --train-size 300 --eval-size 380 \                 --out /content/project/bird_model_official.pth

## 4. Save the resultThe checkpoint carries `class_names`, the split manifest hashes, the full recipe, per-epochhistory and the final test numbers — enough to reproduce or audit the run later.

In [ ]:
import torch, jsonck = torch.load("/content/project/bird_model_official.pth", map_location="cpu")print("val top-1 :", round(ck["val_acc"], 2))print("test top-1:", round(ck.get("test_top1", float('nan')), 2),      "| +hflip TTA:", round(ck.get("test_top1_tta", float('nan')), 2))print("test top-5:", round(ck.get("test_top5", float('nan')), 2))print("used EMA  :", ck.get("used_ema"))print("split hashes:", json.dumps(ck["split_manifest_sha256"], indent=2)[:220], "...")if ck.get("test_top1", 0) > 90:    print("\nWARNING: >90% from this recipe suggests a leak. Investigate before reporting.")

In [ ]:
# Strip nothing - this checkpoint has no optimizer state to begin with.from google.colab import filesfiles.download("/content/project/bird_model_official.pth")

## 5. Back on your machine```bashmv ~/Downloads/bird_model_official.pth ~/Desktop/birdsproject/BIRD_MODEL_PATH=bird_model_official.pth ./run.sh restart```The app prefers a checkpoint carrying `split_manifest_sha256`, and the contamination warningon the evaluation tabs disappears automatically once one is loaded.Then fit the open-set rejection thresholds against *this* model — official val is genuinelyheld out for it, which is what makes the threshold meaningful:```bashpython3 scripts/fit_openset.py --model bird_model_official.pth```